## TUGAS MANDIRI (Dikerjakan Selama 1 Minggu)

> **Tenggat waktu:** dikumpulkan paling lambat **sebelum Pertemuan 12 dimulai**.
> **Sifat tugas:** individu.

### Konteks / Skenario

Manajemen "Kopi Nusantara" (skenario yang sama dari Pertemuan 7, 9, dan 10) ingin memiliki alat bantu **prediksi omzet harian** tiap cabang, agar dapat merencanakan stok bahan baku dengan lebih baik. Anda ditugaskan membangun model regresi untuk memprediksi `total_penjualan` sebuah pesanan berdasarkan atribut-atributnya.

### Menyiapkan Dataset

Jalankan cell berikut untuk membuat dataset.

In [1]:
import numpy as np
import pandas as pd

np.random.seed(111)
n = 2500
menu_list = ["Kopi Susu", "Americano", "Cappuccino", "Roti Bakar", "Kentang Goreng", "Es Teh"]
cabang_list = ["Magelang", "Yogyakarta", "Semarang"]
waktu_list = ["Pagi", "Siang", "Sore", "Malam"]

data_tugas11 = pd.DataFrame({
    "nama_menu": np.random.choice(menu_list, size=n),
    "cabang": np.random.choice(cabang_list, size=n),
    "waktu_pesan": np.random.choice(waktu_list, size=n),
    "qty": np.random.randint(1, 6, size=n),
    "harga_satuan": np.random.choice([15000, 18000, 20000, 22000, 25000, 30000], size=n),
})
data_tugas11["total_penjualan"] = data_tugas11["qty"] * data_tugas11["harga_satuan"] + np.random.normal(0, 8000, size=n)
data_tugas11.to_csv("tugas11_penjualan.csv", index=False)

print(f"Dataset berhasil dibuat: {len(data_tugas11)} baris")
print(data_tugas11.head())

Dataset berhasil dibuat: 2500 baris
        nama_menu      cabang waktu_pesan  qty  harga_satuan  total_penjualan
0  Kentang Goreng    Semarang       Malam    1         30000     27750.163532
1  Kentang Goreng  Yogyakarta        Sore    3         20000     64216.160744
2  Kentang Goreng    Semarang       Siang    5         22000    108722.792592
3  Kentang Goreng  Yogyakarta        Pagi    2         20000     54483.205872
4      Roti Bakar    Magelang       Malam    2         30000     58157.944120


### Instruksi Pengerjaan

Buat notebook baru **`Tugas11_[NPM]_[Nama Lengkap].ipynb`**, buat `SparkSession`, baca `tugas11_penjualan.csv`, lalu kerjakan bagian **A sampai E** berikut:

---

In [5]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

spark = SparkSession.builder \
    .appName("Tugas11-MLlibRegresi") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
spark.conf.set("spark.sql.shuffle.partitions", "4")

# Membaca berkas CSV
df = spark.read.csv("tugas11_penjualan.csv", header=True, inferSchema=True)
print("Tipe objek:", type(df))
df.printSchema()

print("SparkSession siap. Versi Spark:", spark.version)

Tipe objek: <class 'pyspark.sql.dataframe.DataFrame'>
root
 |-- nama_menu: string (nullable = true)
 |-- cabang: string (nullable = true)
 |-- waktu_pesan: string (nullable = true)
 |-- qty: integer (nullable = true)
 |-- harga_satuan: integer (nullable = true)
 |-- total_penjualan: double (nullable = true)

SparkSession siap. Versi Spark: 3.5.9


**A. Feature Engineering** *(bobot 20%)*

Gunakan `StringIndexer` untuk mengubah kolom `nama_menu`, `cabang`, dan `waktu_pesan` menjadi indeks numerik. Gabungkan seluruh fitur (`qty`, `harga_satuan`, dan ketiga kolom hasil indexing) menjadi satu kolom `features` menggunakan `VectorAssembler`.

In [47]:
from pyspark.ml.feature import StringIndexer
from pyspark.ml.feature import VectorAssembler

indexer_menu = StringIndexer(inputCol="nama_menu", outputCol="nama_menu_idx").fit(df)
indexer_cabang = StringIndexer(inputCol="cabang", outputCol="cabang_idx").fit(df)
indexer_waktu = StringIndexer(inputCol="waktu_pesan", outputCol="waktu_pesan_idx").fit(df)

df_indexed = indexer_menu.transform(df)
df_indexed = indexer_cabang.transform(df_indexed)
df_indexed = indexer_waktu.transform(df_indexed)

df_indexed.select("nama_menu", "nama_menu_idx", "cabang", "cabang_idx", "waktu_pesan", "waktu_pesan_idx").show(5)

# Menggabungkan seluruh fitur (qty, harga_satuan, dan ketiga kolom indexing) menjadi 1 kolom
assembler = VectorAssembler(
    inputCols=["qty", "harga_satuan", "nama_menu_idx", "cabang_idx", "waktu_pesan_idx"],
    outputCol="features"
)

df_final = assembler.transform(df_indexed)
df_final.select("features").show(5, truncate=False)

+--------------+-------------+----------+----------+-----------+---------------+
|     nama_menu|nama_menu_idx|    cabang|cabang_idx|waktu_pesan|waktu_pesan_idx|
+--------------+-------------+----------+----------+-----------+---------------+
|Kentang Goreng|          5.0|  Semarang|       2.0|      Malam|            1.0|
|Kentang Goreng|          5.0|Yogyakarta|       1.0|       Sore|            3.0|
|Kentang Goreng|          5.0|  Semarang|       2.0|      Siang|            2.0|
|Kentang Goreng|          5.0|Yogyakarta|       1.0|       Pagi|            0.0|
|    Roti Bakar|          4.0|  Magelang|       0.0|      Malam|            1.0|
+--------------+-------------+----------+----------+-----------+---------------+
only showing top 5 rows

+-------------------------+
|features                 |
+-------------------------+
|[1.0,30000.0,5.0,2.0,1.0]|
|[3.0,20000.0,5.0,1.0,3.0]|
|[5.0,22000.0,5.0,2.0,2.0]|
|[2.0,20000.0,5.0,1.0,0.0]|
|[2.0,30000.0,4.0,0.0,1.0]|
+---------------------

**B. Train/Test Split & Pelatihan Model** *(bobot 20%)*

Bagi data menjadi 75% training dan 25% testing (`seed=101`). Latih model `LinearRegression` dengan `labelCol="total_penjualan"`.


In [48]:
from pyspark.ml.regression import LinearRegression

# bagi data, 75% training, 25% testing
train, test = df_final.randomSplit([0.75, 0.25], seed=101)

lr = LinearRegression(featuresCol="features", labelCol="total_penjualan")
model = lr.fit(train)

print(f"Jumlah data training : {train.count()}")
print(f"Jumlah data testing : {test.count()}")
print("\nModel berhasil dilatih!")
print("Koefisien tiap fitur:", model.coefficients)
print("Intercept:", model.intercept)

Jumlah data training : 1839
Jumlah data testing : 661

Model berhasil dilatih!
Koefisien tiap fitur: [22046.663271025755,3.0229208681606696,-123.21425598657828,279.2845246167609,339.63879532317213]
Intercept: -66903.1776648588


**C. Evaluasi Model** *(bobot 20%)*

Hitung dan tampilkan **RMSE** dan **R²** dari model pada data testing. Tulis 1-2 kalimat interpretasi: apakah model ini cukup baik untuk dipakai memprediksi omzet?


In [49]:
from pyspark.ml.evaluation import RegressionEvaluator

predictions = model.transform(test)
predictions.select("qty", "harga_satuan", "nama_menu_idx", "cabang_idx", "waktu_pesan_idx", "prediction").show(5)

# RMSE
eval_rmse = RegressionEvaluator(labelCol="total_penjualan", predictionCol="prediction", metricName="rmse")
rmse = eval_rmse.evaluate(predictions)

# R-squared
eval_r2 = RegressionEvaluator(labelCol="total_penjualan", predictionCol="prediction", metricName="r2")
r2 = eval_r2.evaluate(predictions)

print(f"RMSE : Rp. {rmse:,.0f}")
print(f"R2   : {r2:.4f} ({r2*100:.1f}% variasi data berhasil dijelaskan model)")

+---+------------+-------------+----------+---------------+------------------+
|qty|harga_satuan|nama_menu_idx|cabang_idx|waktu_pesan_idx|        prediction|
+---+------------+-------------+----------+---------------+------------------+
|  1|       25000|          0.0|       0.0|            1.0|31056.146105506865|
|  1|       25000|          0.0|       0.0|            1.0|31056.146105506865|
|  4|       22000|          0.0|       0.0|            1.0| 88127.37331410211|
|  4|       22000|          0.0|       0.0|            1.0| 88127.37331410211|
|  4|       30000|          0.0|       0.0|            1.0|112310.74025938749|
+---+------------+-------------+----------+---------------+------------------+
only showing top 5 rows

RMSE : Rp. 10,901
R2   : 0.9107 (91.1% variasi data berhasil dijelaskan model)


Model ini dinilai sudah cukup baik untuk memprediksi omzet karena tingkat akurasinya tinggi, di mana 91,1% variasi data berhasil dijelaskan oleh model (R²). Selain itu, rata-rata kesalahan prediksinya juga tergolong kecil, yaitu hanya sekitar Rp 10.901.

**D. Eksperimen Perbandingan Fitur** *(bobot 25%)*

Latih **model kedua** yang HANYA menggunakan fitur `qty` dan `harga_satuan` (tanpa informasi menu/cabang/waktu). Bandingkan R² model kedua ini dengan model utama pada bagian C. Tulis kesimpulan: apakah menambahkan informasi menu/cabang/waktu benar-benar meningkatkan akurasi prediksi secara berarti?


In [50]:
assembler_sederhana = VectorAssembler(
    inputCols=["qty", "harga_satuan"],
    outputCol="features_sederhana"
)
df_final2 = assembler_sederhana.transform(df_final)
df_final2.select("qty", "harga_satuan", "features_sederhana").show(5, truncate=False)

train2, test2 = df_final2.randomSplit([0.75, 0.25], seed=101)

lr2 = LinearRegression(featuresCol="features_sederhana", labelCol="total_penjualan")
model2 = lr2.fit(train2)

pred2 = model2.transform(test2)
pred2.select("qty", "harga_satuan", "prediction").show(5)

eval_r2_2 = RegressionEvaluator(labelCol="total_penjualan", predictionCol="prediction", metricName="r2")
r2_2 = eval_r2_2.evaluate(pred2)
print(f"R²   : {r2_2:.4f}  ({r2_2*100:.1f}% variasi data berhasil dijelaskan model)")

+---+------------+------------------+
|qty|harga_satuan|features_sederhana|
+---+------------+------------------+
|1  |30000       |[1.0,30000.0]     |
|3  |20000       |[3.0,20000.0]     |
|5  |22000       |[5.0,22000.0]     |
|2  |20000       |[2.0,20000.0]     |
|2  |30000       |[2.0,30000.0]     |
+---+------------+------------------+
only showing top 5 rows

+---+------------+------------------+
|qty|harga_satuan|        prediction|
+---+------------+------------------+
|  1|       25000|31177.718851858037|
|  1|       25000|31177.718851858037|
|  4|       22000| 88260.08378640363|
|  4|       22000| 88260.08378640363|
|  4|       30000|112428.48226630974|
+---+------------+------------------+
only showing top 5 rows

R²   : 0.9108  (91.1% variasi data berhasil dijelaskan model)


Berdasarkan hasil eksperimen, nilai R² model sederhana yang hanya menggunakan fitur qty dan harga_satuan adalah sebesar 0.9108 (91,1%). Nilai ini hampir identik dengan model utama yang menggunakan fitur lengkap (R² = 0.9107). Dapat disimpulkan bahwa penambahan informasi menu, cabang, dan waktu pesan TIDAK memberikan peningkatan akurasi prediksi yang berarti (signifikan). Model prediksi omzet ternyata sudah sangat baik hanya dengan mengandalkan kuantitas barang (qty) dan harga satuan saja.

**E. Prediksi Skenario Baru** *(bobot 15%)*

Buat sebuah DataFrame baru berisi **3 skenario pesanan (data dummy)** (boleh bebas kombinasi menu/cabang/waktu/qty/harga, namun harus melalui tahapan `StringIndexer` dan `VectorAssembler` yang sama seperti data training — perhatikan: gunakan model `StringIndexer` yang **sama** yang sudah di-`fit()` sebelumnya, jangan membuat `StringIndexer` baru). Tampilkan hasil prediksi `total_penjualan` untuk ketiga skenario tersebut.

In [51]:
# Data Dummy
skenario_pesanan = [
    ("Americano", "Magelang", "Malam", 5, 12000),
    ("Cappuccino", "Yogyakarta", "Pagi", 2, 14000),
    ("Roti Bakar", "Semarang", "Siang", 5, 10000)
]
kolom = ["nama_menu", "cabang", "waktu_pesan", "qty", "harga_satuan"]

df_dummy = spark.createDataFrame(skenario_pesanan, kolom)
df_dummy.show()

# StringIndexer yang sudah di fit
df_indexed_dummy = indexer_menu.transform(df_dummy)
df_indexed_dummy = indexer_cabang.transform(df_indexed_dummy)
df_indexed_dummy = indexer_waktu.transform(df_indexed_dummy)
df_indexed_dummy.show()

# VectorAssembler
df_final_dummy = assembler.transform(df_indexed_dummy)
df_final_dummy.show()

# Prediksi total_penjualan
pred_dummy = model.transform(df_final_dummy)
pred_dummy.select("nama_menu", "cabang", "waktu_pesan", "qty", "harga_satuan", "prediction").show(5, truncate=False)

+----------+----------+-----------+---+------------+
| nama_menu|    cabang|waktu_pesan|qty|harga_satuan|
+----------+----------+-----------+---+------------+
| Americano|  Magelang|      Malam|  5|       12000|
|Cappuccino|Yogyakarta|       Pagi|  2|       14000|
|Roti Bakar|  Semarang|      Siang|  5|       10000|
+----------+----------+-----------+---+------------+

+----------+----------+-----------+---+------------+-------------+----------+---------------+
| nama_menu|    cabang|waktu_pesan|qty|harga_satuan|nama_menu_idx|cabang_idx|waktu_pesan_idx|
+----------+----------+-----------+---+------------+-------------+----------+---------------+
| Americano|  Magelang|      Malam|  5|       12000|          0.0|       0.0|            1.0|
|Cappuccino|Yogyakarta|       Pagi|  2|       14000|          2.0|       1.0|            0.0|
|Roti Bakar|  Semarang|      Siang|  5|       10000|          4.0|       2.0|            2.0|
+----------+----------+-----------+---+------------+------------